In [8]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("indexicality_batch_results.csv")

# ensure correct column naming (DO NOT rely on CSV order)
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "interaction_mode": "interaction",
    "timing_mode": "timing"
})

df["stable"] = df["stable"].astype(bool)

# =========================================================
# IDENTIFY STATE COLUMNS (IMPORTANT FOR YOUR THEORY)
# =========================================================
state_cols = [c for c in df.columns if c.startswith("count_state_")]

# =========================================================
# HELPERS
# =========================================================
def mean_stable_time(x):
    sub = df.loc[x.index]
    stable = sub[sub["stable"]]
    return stable["time_to_stability"].mean() if len(stable) > 0 else np.nan

def sd_stable_time(x):
    sub = df.loc[x.index]
    stable = sub[sub["stable"]]
    return stable["time_to_stability"].std() if len(stable) > 0 else np.nan

# =========================================================
# CORE TABLE
# =========================================================
table = df.groupby(["K", "interaction", "timing"]).agg(
    # 1. SYSTEM-LEVEL INDEXICAL STATE
    mean_state=("final_mean", "mean"),
    sd_state=("final_mean", "std"),

    # 2. STABILITY (CRITICAL FOR YOUR RQ)
    stability_rate=("stable", "mean"),

    # 3. TIME TO STABILITY
    mean_time=("time_to_stability", mean_stable_time),
    sd_time=("time_to_stability", sd_stable_time),

    # 4. SAMPLE SIZE
    n_runs=("stable", "count"),

    # 5. INDEXICAL DISTRIBUTION (IMPORTANT EXTENSION)
    **{col: (col, "mean") for col in state_cols}
).reset_index()

# =========================================================
# FORMATTING FOR PAPER
# =========================================================
table["State (M SD)"] = table.apply(
    lambda r: f"{r['mean_state']:.2f} ({r['sd_state']:.2f})",
    axis=1
)

table["Stability (%)"] = table["stability_rate"].apply(lambda x: f"{x*100:.1f}%")

table["Time (M SD)"] = table.apply(
    lambda r: (
        "NA" if pd.isna(r["mean_time"])
        else f"{r['mean_time']:.1f} ({r['sd_time']:.1f})"
    ),
    axis=1
)

# =========================================================
# FINAL OUTPUT TABLE
# =========================================================
final_table = table[
    ["K", "interaction", "timing",
     "State (M SD)",
     "Stability (%)",
     "Time (M SD)",
     "n_runs"] + state_cols
]

final_table = final_table.sort_values(["K", "interaction", "timing"])

final_table

,K,interaction,timing,State (M SD),Stability (%),Time (M SD),n_runs,count_state_0,count_state_1,count_state_2,count_state_3,count_state_4,count_state_5,count_state_6,count_state_7,count_state_8,count_state_9
0,3,dyadic,asynchronous,1.55 (0.16),100.0%,23.0 (11.0),30,7.066667,84.800000,99.800000,8.333333,NaN,NaN,NaN,NaN,NaN,NaN
1,3,dyadic,partial,1.53 (0.16),100.0%,25.5 (11.2),30,5.266667,89.433333,99.000000,6.300000,NaN,NaN,NaN,NaN,NaN,NaN
2,3,dyadic,synchronous,1.50 (0.18),100.0%,23.1 (11.1),30,8.233333,90.700000,94.266667,6.800000,NaN,NaN,NaN,NaN,NaN,NaN
3,3,group,asynchronous,1.46 (0.20),100.0%,81.9 (50.7),30,8.666667,95.333333,90.566667,5.433333,NaN,NaN,NaN,NaN,NaN,NaN
4,3,group,partial,1.51 (0.21),93.3%,103.8 (78.2),30,6.533333,94.166667,90.900000,8.400000,NaN,NaN,NaN,NaN,NaN,NaN
5,3,group,synchronous,1.55 (0.20),93.3%,97.5 (77.4),30,4.633333,89.666667,95.933333,9.766667,NaN,NaN,NaN,NaN,NaN,NaN
6,5,dyadic,asynchronous,2.55 (0.18),100.0%,26.8 (11.8),30,0.466667,12.300000,79.966667,91.533333,14.866667,0.866667,NaN,NaN,NaN,NaN
7,5,dyadic,partial,2.53 (0.21),100.0%,28.6 (11.8),30,1.100000,13.033333,81.733333,88.733333,14.566667,0.833333,NaN,NaN,NaN,NaN
8,5,dyadic,synchronous,2.45 (0.20),100.0%,26.7 (12.3),30,0.800000,18.266667,86.700000,79.633333,14.100000,0.500000,NaN,NaN,NaN,NaN
9,5,group,asynchronous,2.53 (0.24),93.3%,106.0 (74.6),30,0.166667,7.466667,88.666667,93.300000,10.233333,0.166667,NaN,NaN,NaN,NaN
